# Concept Transformer (CVIT) on CUB-200-2011 — Colab / Apple-Silicon Mac walkthrough

**Colab:** `Runtime → Change runtime type → GPU (A100/T4)`, then `Runtime → Run all`.

**Apple-Silicon Mac (M1–M4) — run locally in Jupyter:** use Python 3.10/3.11 (not 3.13) in a fresh virtualenv:
```bash
python3.11 -m venv ctc-env && source ctc-env/bin/activate
pip install jupyterlab torch torchvision numpy pandas matplotlib
jupyter lab        # open this notebook, then Run All
```
The notebook auto-detects the Apple GPU (`mps`), patches the repo to train on it in fp32, and sets `NUM_WORKERS = 0`
(needed on macOS). Expect it to be much slower than an A100: use the small defaults (5 epochs, batch 32) or `DEBUG = True` first.

Everything is configurable in the next cell. Paths are on Colab's local disk (`/content`) for speed;
set `SAVE_TO_DRIVE = True` to copy logs, checkpoints and figures to Google Drive at the end (Colab VMs are ephemeral).

## 1. Introduction

**Paper:** *Concept Transformers: Interpretable-by-design transformers* (Rigotti et al., ICLR 2022, IBM Research).
A Concept Transformer (CT) replaces the usual classification head with a cross-attention layer whose *keys/values are
human-defined concepts* (here CUB bird attributes such as "has_wing_color::black"). The model classifies from the
concept-attended representation, so the attention weights **are** the explanation. An extra **explanation loss**
(`--expl_lambda`) pushes the attention towards ground-truth attribute annotations, and a sparsity penalty
(`--attention_sparsity`) keeps the attention concise. In `cvit_cub.py` the backbone is a pretrained ViT-B/16 (timm);
the 14x14 patch tokens attend to *spatial* concepts (attributes tied to a body part) and the CLS token attends to
*non-spatial* concepts (e.g. overall size/shape/colour).

**Code:** `cvit_cub.py` → `ctc.run_exp` builds a PyTorch-Lightning `CTCModel` + `CUB2011Parts` datamodule, trains with
`val_ce_loss`-based checkpointing, then calls `trainer.test()`.

In [ ]:
# ---------------- Configuration (edit me) ----------------
REPO_URL     = "https://github.com/IBM/concept_transformer"
import os, sys
IN_COLAB     = "google.colab" in sys.modules
WORK_DIR     = "/content" if IN_COLAB else os.path.expanduser("~/ctc_work")   # local Mac: ~/ctc_work
os.makedirs(WORK_DIR, exist_ok=True)
REPO_DIR     = f"{WORK_DIR}/concept_transformer"
DATA_DIR     = f"{WORK_DIR}/data/cub2011"        # passed as --data_dir ; images end up in DATA_DIR/CUB_200_2011/images

# Training hyper-parameters (repo defaults: 50 epochs, bs 32, warmup 10, lr 5e-5)
MAX_EPOCHS   = 5      # T4 preset (A100: 10-15; paper: 50)
BATCH_SIZE   = 32     # T4 (16 GB) preset; A100 can use 64
WARMUP       = 1       # short run -> short warm-up (repo default 10)
NUM_WORKERS  = 2 if IN_COLAB else 0   # Colab has 2 CPUs; keep 0 on macOS
LEARNING_RATE= 5e-5
DEBUG        = False   # True = one batch only (fast_dev_run) to smoke-test the pipeline

SAVE_TO_DRIVE = True    # mounts Drive in the last cell so results survive the session
DRIVE_OUT_DIR = "/content/drive/MyDrive/concept_transformer_run"

In [ ]:
import platform, sys, subprocess
print("OS     :", platform.platform())
print("Python :", sys.version)
import torch as _t
print("MPS (Apple GPU) available:", _t.backends.mps.is_available())
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "No NVIDIA GPU found")

## 2. Environment setup

### Compatibility problem (read this)
The repo's `requirements.txt` pins `torch==1.10.0`, `pytorch-lightning==1.4.8`, `numpy==1.22.0`, `pandas==1.3.3`, ... and the
trainer asks for **NVIDIA Apex** (`amp_backend="apex"`). Current Colab runs Python 3.11/3.12 with torch 2.x, for which:

| Issue | Why it breaks | Resolution used here |
|---|---|---|
| `torch==1.10.0` / `torchvision==0.11.1` / `numpy==1.22` / `pandas==1.3.3` | no wheels for Python ≥ 3.11 | keep Colab's preinstalled torch/torchvision/numpy/pandas (CUDA build matched to the A100) |
| `pytorch-lightning==1.4.8` | imports removed torch internals under torch 2.x | `pytorch-lightning==1.9.5` (last 1.x; keeps `gpus=`, `auto_select_gpus`, same checkpoint API) |
| `torchmetrics==0.5` | pairs with old PL | `torchmetrics==0.11.4` (still accepts `accuracy(preds, target)`) |
| `lightning-bolts==0.3.4` (only used for `LinearWarmupCosineAnnealingLR`) | pulls an old PL | not installed; the scheduler is re-implemented inline (identical schedule) |
| `albumentations==1.0.3` | old numpy/opencv ABI; newer versions moved `HorizontalFlip`/`Resize`/... out of the submodules the repo imports from | `albumentations==1.4.24` + datamodule imports patched to `from albumentations import ...` |
| `timm==0.4.12` | – | kept: pure python, works on torch 2 (model code subclasses its `VisionTransformer`) |
| NumPy 2 (`np.Inf` removed) | PL 1.9.5 calls `np.Inf` in `ModelCheckpoint` | alias shim added to `ctc/__init__.py` (`np.Inf = np.inf`, ...) |
| `amp_backend="apex"` | Apex not installable on Colab | use PL native AMP `precision=16` |
| `weights_summary`, `progress_bar_refresh_rate` Trainer args | removed in PL 1.7 | removed from the trainer calls |
| `Image.ANTIALIAS` in `viz_utils.py` | removed in Pillow 10 | `Image.LANCZOS` |
| `epyc` in requirements | only used for MNIST scaling sweeps | not needed |

In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!pip -q install pytorch-lightning==1.9.5 torchmetrics==0.11.4 timm==0.4.12 albumentations==1.4.24 tensorboard

### Patch the repo for modern PyTorch / Lightning (idempotent)

In [ ]:
import re, pathlib

def patch(path, pairs, regex=False):
    p = pathlib.Path(path); s = p.read_text(); orig = s
    for a, b in pairs:
        s = re.sub(a, b, s) if regex else s.replace(a, b)
    if s != orig:
        p.write_text(s); print("patched", path)
    else:
        print("already patched / no change", path)

# 1) ctc_model.py: drop pl_bolts, drop apex, drop args removed in PL 1.7
patch("ctc/ctc_model.py", [
    ("from pl_bolts.optimizers.lr_scheduler import LinearWarmupCosineAnnealingLR\n",
     """import math
from torch.optim.lr_scheduler import LambdaLR


def LinearWarmupCosineAnnealingLR(optimizer, warmup_epochs, max_epochs, warmup_start_lr=0.0, eta_min=0.0):
    # Same schedule as pl_bolts: linear warm-up then cosine decay (per-epoch stepping), re-implemented to avoid lightning-bolts.
    base = optimizer.param_groups[0]["lr"]
    def f(epoch):
        if warmup_epochs > 0 and epoch < warmup_epochs:
            return (warmup_start_lr + (base - warmup_start_lr) * epoch / max(1, warmup_epochs - 1)) / base
        t = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
        return (eta_min + 0.5 * (base - eta_min) * (1 + math.cos(math.pi * t))) / base
    return LambdaLR(optimizer, f)
"""),
    ('kwargs = {"amp_backend": "apex", "amp_level": "O2", "precision": 16}', 'kwargs = {"precision": 16}'),
    ("            weights_summary=\"full\",\n", ""),
    ("                progress_bar_refresh_rate=10,\n", ""),
    ("            progress_bar_refresh_rate=10,\n", ""),
])

# 1a) NumPy 2 removed aliases (np.Inf, np.NaN, np.float_) that pytorch-lightning 1.9.5 still uses -> restore them before PL is used
if "import numpy as _np" not in pathlib.Path("ctc/__init__.py").read_text():
  patch("ctc/__init__.py", [("from .ctc import concepts_cost,", """import numpy as _np
for _old, _new in (("Inf", "inf"), ("NaN", "nan"), ("float_", "float64")):
    if not hasattr(_np, _old):
        setattr(_np, _old, getattr(_np, _new))

from .ctc import concepts_cost,""")])

# 1b) datamodule: albumentations >=1.4 moved HorizontalFlip/Normalize/Resize/Rotate; import them from the top-level package
patch("data/cub2011parts_datamodule.py", [
    ("from albumentations.augmentations.geometric.resize import Resize\n", "from albumentations import Resize\n"),
    ("from albumentations.augmentations.geometric.rotate import Rotate\n", "from albumentations import Rotate\n"),
    ("from albumentations.augmentations.transforms import HorizontalFlip, Normalize\n", "from albumentations import HorizontalFlip, Normalize\n"),
])

# 1c) Apple-Silicon: trainer branch for MPS (fp32; apex/fp16 are CUDA-only)
patch("ctc/ctc_model.py", [("""        else:
            trainer = pl.Trainer(
                logger=logger,
                max_epochs=max_epochs,
                callbacks=callbacks,
            )""", """        else:
            extra = dict(accelerator="mps", devices=1) if torch.backends.mps.is_available() else {}
            trainer = pl.Trainer(
                logger=logger,
                max_epochs=max_epochs,
                callbacks=callbacks,
                gradient_clip_val=1.0,
                **extra,
            )""")])

# 2) viz_utils.py: Pillow>=10 and configurable attribute root
patch("viz_utils.py", [("Image.ANTIALIAS", "Image.LANCZOS")])
patch("viz_utils.py", [("def load_attributes(root='/data/Datasets/'):", "def load_attributes(root=os.environ.get('CUB_ROOT', '/data/Datasets/')):")])
!grep -n "progress_bar_refresh_rate\|weights_summary\|apex" ctc/ctc_model.py || echo "no leftover removed args"

### Verify PyTorch / device

In [ ]:
import torch, torchvision, pytorch_lightning as pl, timm, albumentations, numpy, pandas
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| PL", pl.__version__,
      "| timm", timm.__version__, "| albumentations", albumentations.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Running on GPU:", torch.cuda.get_device_name(0), "| CUDA", torch.version.cuda)
    x = torch.randn(4096, 4096, device="cuda"); print("matmul OK:", (x @ x).mean().item() is not None)
elif torch.backends.mps.is_available():
    print("Running on Apple GPU (MPS)")
    x = torch.randn(2048, 2048, device="mps"); print("matmul OK:", (x @ x).mean().item() is not None)
else:
    print("!! Running on CPU only - training will be extremely slow; use DEBUG=True or a GPU/MPS machine")

## 3. Data download & preparation

* **Source:** official CUB-200-2011 page <https://www.vision.caltech.edu/datasets/cub_200_2011/> (the archive is hosted at
  <https://data.caltech.edu/records/65de6-vp158>, file `CUB_200_2011.tgz`, ~1.1 GB). It contains images, `images.txt`,
  `image_class_labels.txt`, `train_test_split.txt`, `bounding_boxes.txt`, `parts/part_locs.txt`, `classes.txt`, and the
  attribute annotations (`attributes/` plus a top-level `attributes.txt` listing the 312 attribute names).
* The repo's own downloader points at `data.deepai.org`, which is unreliable, so we download from Caltech ourselves and the
  dataset class then finds everything already in place ("Files already downloaded and verified").
* **Expected layout** (what `CUB2011Parts_dataset` reads):
  `DATA_DIR/CUB_200_2011/{images,parts,attributes,images.txt,classes.txt,...,attributes.txt}`.
* **Preprocessing done automatically at first run:** move `attributes.txt` into `CUB_200_2011/`, crop every image to its
  bounding box into `CUB_200_2011/cropped/` (≈11.8k images), filter to 112 retained attributes, map attributes to
  14x14 patches via part locations. Training uses Resize 224, flip, rotate ±30°.

In [ ]:
import os, tarfile, shutil
os.makedirs(DATA_DIR, exist_ok=True)
TGZ = f"{DATA_DIR}/CUB_200_2011.tgz"
URLS = ["https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1",
        "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"]

if not os.path.isdir(f"{DATA_DIR}/CUB_200_2011/images"):
    if not (os.path.exists(TGZ) and os.path.getsize(TGZ) > 1e9):
        for u in URLS:
            rc = os.system(f'curl -L --fail -o "{TGZ}" "{u}"')
            if rc == 0 and os.path.getsize(TGZ) > 1e9: break
        else:
            raise SystemExit("Download failed. Download CUB_200_2011.tgz manually from "
                             "https://www.vision.caltech.edu/datasets/cub_200_2011/ and upload it to " + DATA_DIR)
    with tarfile.open(TGZ) as t: t.extractall(DATA_DIR)

# attributes.txt sits at the archive root; the code expects it inside CUB_200_2011/
if os.path.exists(f"{DATA_DIR}/attributes.txt"):
    shutil.move(f"{DATA_DIR}/attributes.txt", f"{DATA_DIR}/CUB_200_2011/attributes.txt")
os.environ["CUB_ROOT"] = DATA_DIR

In [ ]:
# Directory structure + verification
!ls {DATA_DIR} {DATA_DIR}/CUB_200_2011
import glob
imgs = glob.glob(f"{DATA_DIR}/CUB_200_2011/images/*/*.jpg")
print("images:", len(imgs), "(expected 11788)")
print("classes:", len(os.listdir(f"{DATA_DIR}/CUB_200_2011/images")), "(expected 200)")
for f in ["images.txt","image_class_labels.txt","train_test_split.txt","bounding_boxes.txt","classes.txt","attributes.txt","parts/part_locs.txt"]:
    print(f"{f:28s}", "OK" if os.path.exists(f"{DATA_DIR}/CUB_200_2011/{f}") else "MISSING")

### Fix malformed lines in `image_attribute_labels.txt`
A handful of lines in this official file have extra/blank-separated fields (e.g. line 709498 has 7 fields instead of 5). Old pandas
tolerated that; current pandas raises `ParserError`. We rewrite the file keeping the first 5 whitespace-separated fields
(`img_id attr_id is_present certainty time`; only the first four are used by the repo). The original is kept as `.orig`.

In [ ]:
attr_file = f"{DATA_DIR}/CUB_200_2011/attributes/image_attribute_labels.txt"
if not os.path.exists(attr_file + ".orig"):
    shutil.copy(attr_file, attr_file + ".orig")
bad = 0
with open(attr_file + ".orig") as fin, open(attr_file, "w") as fout:
    for i, line in enumerate(fin, 1):
        t = line.split()
        if len(t) != 5:
            bad += 1
            if bad <= 5: print(f"line {i}: {len(t)} fields -> {line.rstrip()!r}")
        if len(t) >= 4:
            t = (t + ["0"])[:5] if len(t) < 5 else t[:5]
            fout.write(" ".join(t) + "\n")
print("malformed lines fixed:", bad)

In [ ]:
# Instantiate the dataset once: this runs the repo's integrity check and the bounding-box cropping (one-off, a few minutes)
import sys; sys.path.insert(0, REPO_DIR)
from data import CUB2011Parts_dataset
# The repo's _check_integrity() swallows the real exception, so run the metadata loader directly to see it.
import traceback
try:
    _probe = object.__new__(CUB2011Parts_dataset); _probe.root = DATA_DIR; _probe.train = True
    CUB2011Parts_dataset._load_metadata(_probe)
    print("metadata OK")
except Exception:
    traceback.print_exc()
    !ls -R {DATA_DIR}/CUB_200_2011 | head -40
    !head -3 {DATA_DIR}/CUB_200_2011/attributes.txt {DATA_DIR}/CUB_200_2011/attributes/image_attribute_labels.txt
    raise
train_ds = CUB2011Parts_dataset(root=DATA_DIR, train=True,  download=False)
test_ds  = CUB2011Parts_dataset(root=DATA_DIR, train=False, download=False)
print("train:", len(train_ds), "test:", len(test_ds), "| classes:", train_ds.num_classes,
      "| spatial concepts:", train_ds.num_spatial_attributes, "| global concepts:", train_ds.num_non_spatial_attributes)

## 4. Training run

Command (executed from the repo directory, log tee'd to `train.log`). Reasoning for non-default values:
`--batch_size 32` (fits a 16 GB T4 with fp16 AMP), `--max_epochs 5` and `--warmup 1` (much shorter than the
paper's 50/10 epochs — raise `MAX_EPOCHS` if you have time/compute), `--num_workers 2` (Colab T4 VMs have 2 CPUs).

In [ ]:
flags = f"--data_dir {DATA_DIR} --max_epochs {MAX_EPOCHS} --batch_size {BATCH_SIZE} --warmup {WARMUP} " \
        f"--num_workers {NUM_WORKERS} --learning_rate {LEARNING_RATE}" + (" --debug" if DEBUG else "")
cmd = f"python cvit_cub.py {flags}"
print(cmd)

In [ ]:
%cd {REPO_DIR}
%env PYTORCH_ENABLE_MPS_FALLBACK=1
!rm -rf logs cub_cvit
!{cmd} 2>&1 | tee train.log

### Training curves (read from the TensorBoard logs Lightning wrote)

In [ ]:
import glob, matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
os.makedirs("outputs", exist_ok=True)
ev_files = sorted(glob.glob("logs/*/version_*"))
if not ev_files:
    print("No logs found (DEBUG mode logs nothing meaningful).")
else:
    ea = EventAccumulator(ev_files[-1]); ea.Reload()
    tags = ea.Tags()["scalars"]; print(tags)
    def curve(t): s = ea.Scalars(t); return [x.step for x in s], [x.value for x in s]
    groups = {"Loss": ["train_loss","train_ce_loss","val_ce_loss"], "Accuracy": ["train_acc","val_acc"],
              "Explanation / sparsity loss": ["train_expl_loss","val_expl_loss","train_l1_loss","val_l1_loss"]}
    fig, axs = plt.subplots(1, 3, figsize=(17, 4))
    for ax, (title, ts) in zip(axs, groups.items()):
        for t in ts:
            if t in tags: ax.plot(*curve(t), label=t, marker="." if "val" in t else None)
        ax.set_title(title); ax.set_xlabel("step"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig("outputs/training_curves.png", dpi=150); plt.show()

## 5. Results & visualization

In [ ]:
# Final test metrics printed by trainer.test() at the end of train.log
import re
log = open("train.log").read()
m = re.findall(r"DATALOADER:0 TEST RESULTS.*?\}", log, flags=re.S)
print(m[-1] if m else log[-2000:])

In [ ]:
# Reload the best (lowest val_ce_loss) checkpoint and evaluate / collect concept attention for every test image
from pytorch_lightning import Trainer
from ctc import load_exp
from viz_utils import batch_predict_results, plot_cub_expl

ckpts = glob.glob("cub_cvit/**/*.ckpt", recursive=True); print("checkpoints:", ckpts)
model, data_module = load_exp(ckpts[0])
model.eval()
accel = dict(accelerator="gpu", devices=1) if torch.cuda.is_available() else \
        dict(accelerator="mps", devices=1) if torch.backends.mps.is_available() else dict(accelerator="cpu")
raw = Trainer(logger=False, enable_checkpointing=False, **accel).predict(model, data_module)
results = batch_predict_results(raw)
acc = results["correct"].float().mean().item()
print(f"Best-checkpoint test accuracy: {acc*100:.2f}%  ({int(results['correct'].sum())}/{len(results['correct'])})")

In [ ]:
# Concept-based explanations: bright patches = patches with attention > 0.6 to some spatial concept;
# listed attributes = spatial concepts (attention > 0.3 on those patches) and global concepts (attention > 0.2).
import numpy as np
correct_idx = np.where(results["correct"].numpy())[0][:3]
wrong_idx   = np.where(~results["correct"].numpy())[0][:3]
print("######## CORRECT PREDICTIONS ########")
for i in correct_idx: plot_cub_expl(results, int(i), data_module)
print("######## WRONG PREDICTIONS ########")
for i in wrong_idx:   plot_cub_expl(results, int(i), data_module)

### Interpretation guide
* **Bright image patches** are where the model's patch tokens attend strongly to a *spatial concept* (e.g. "has_bill_shape::hooked" on the bill).
  If the bright region sits on the bird and on the body part the attribute names, the explanation is faithful/plausible.
* **Spatial / global explanations** are the attributes carrying most attention mass for that image. For a correct prediction these
  should be attributes that really distinguish the predicted species; for a wrong one they often reveal *why* (e.g. the attended
  attributes match a visually similar species).
* `expl_lambda` trades accuracy vs. agreement with the ground-truth attributes; `attention_sparsity` makes the explanations shorter.

In [ ]:
# Save everything (logs, curves, best checkpoint) to Google Drive so it survives the session
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    !cp -r train.log outputs logs cub_cvit {DRIVE_OUT_DIR}/
    print("saved to", DRIVE_OUT_DIR)

## 6. Discussion

*(Fill in after your run — a few things worth checking and commenting on)*

* **Challenges:** dependency pins (torch 1.10 / PL 1.4.8 / Apex) are unusable on current Colab → see the table in §2; the repo's downloader URL (`data.deepai.org`) is unreliable → downloaded from Caltech instead; hard-coded `/data/Datasets/` in `viz_utils.load_attributes`.
* **What worked / didn't:** report final test accuracy vs. the paper (~ 85–90% for the ViT-B CVIT on CUB), runtime per epoch on the A100, and whether a short schedule (`MAX_EPOCHS`) left accuracy below the paper.
* **Possible extensions:**
  * Run the `--baseline` (no concepts) and `--expl_lambda 0` variants to measure the accuracy cost/benefit of the explanation loss.
  * Sweep `--attention_sparsity` and `--expl_lambda` and quantify explanation quality (e.g. precision/recall of attended attributes vs. ground truth).
  * Use bf16 instead of fp16 on A100 (`precision="bf16"`) and a larger backbone (ViT-L) for higher accuracy.
  * Learn extra unsupervised concepts (`n_unsup_concepts`) and inspect what they capture.
  * Add a human-evaluation or intervention test: edit concept attention at test time and measure the effect on predictions.